# nb07_blend : OOF 기반 앙상블·대조 분석 (경량, 수 분)

- **가속기 : None (CPU)** · **인터넷 : ON** (kagglehub로 Nb04_train 버전별 OOF 다운로드)
- **입력** : Dataset `hms-processed`(meta.csv만 사용), `hms-src`, 노트북 `Nb03` Output(p1_oof.csv)
- **출력** : `/kaggle/working/blend/` (비교표 CSV, results.csv 추가 행)
- **목적** : 지표 재계산(nb03, 약 3시간) 없이 저장된 OOF만으로 앙상블·대조 실험을 반복

| 섹션 | 내용 | 판정 |
|---|---|---|
| 0 | 설정, 입력 탐색 | - |
| 1 | OOF 수집 : P1v2 + E1~E5 (Input → 업로드 파일 → kagglehub 순으로 탐색) | 정렬 점검 |
| 2 | 단독 모델 표 + **클래스별 고득표 KL** + 편측↔전반 혼동률 | H3 (E5 vs E4), **H4 (E6 vs E4)** |
| 3 | **나중 결합 대조** : 0.5·E2 + 0.5·E3 vs E4 | 초기 결합 효과 |
| 4 | **딥러닝 + P1v2 평균** (가중치 0.5 사전 지정) + 보정 대조 | E8 사전 확인 |
| 6 | **E8 후보 비교** : 딥러닝 묶음 × (원본 / +T / +P1v2 / +P1v2+T), 가중치·T 중첩 선택 | E8 선택, 제출 레시피 |
| 7 | 저장 | results.csv 행 |

## 0. 설정 (목적 : 입력 탐색 / 출력 : SRC_ROOT, META, OUT_DIR)

In [ ]:
import sys, os, json, time, signal, socket
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("HMS_INPUT_ROOT", "/kaggle/input"))

def find_file(*patterns):
    for pat in patterns:
        hits = sorted(INPUT_ROOT.glob(pat)) if INPUT_ROOT.exists() else []
        if hits:
            return hits[0]
    return None

SRC_ROOT = Path(os.environ.get("HMS_SRC_DIR") or find_file("**/src/eval/metrics.py").parents[2])
sys.path.insert(0, str(SRC_ROOT))
META_PATH = Path(os.environ.get("HMS_META") or find_file("**/hms_processed/meta.csv", "**/meta.csv"))

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from src.data.targets import PROB_COLS, CLASSES
from src.eval.metrics import evaluate, kl_divergence

OUT_DIR = Path(os.environ.get("HMS_OUT_DIR", "/kaggle/working")) / "blend"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PRED = [f"pred_{c}" for c in CLASSES]
TGT = PROB_COLS + ["axis_lat", "axis_pd", "axis_mask", "y_seizure_bin"]
EXPS = ["E1", "E2", "E3", "E4", "E5", "E6"]                    # 없는 실험은 자동으로 건너뜀
NB04_HANDLE = "sungik4057/nb04-train"
NB04_VERSIONS = range(20, 0, -1)                         # 확인할 Nb04_train 버전 (최신부터 : 같은 실험을 다시 돌렸으면 최신 결과 사용)
FETCH_TIMEOUT = 60                                       # 버전 1개당 최대 대기(초)
VERSION_HINT = {"E1": 1, "E2": 3, "E3": 5, "E4": 6, "E5": 7}  # 확인된 Nb04_train 버전 (results.csv) : 먼저 시도

meta = pd.read_csv(META_PATH)
base = meta[["eeg_id", "patient_id", "fold", "high_vote", "expert_consensus"] + TGT].copy()
print("SRC_ROOT :", SRC_ROOT, "\nMETA     :", META_PATH, f"({len(base):,}건)", "\nOUT_DIR  :", OUT_DIR)

## 1. OOF 수집 (목적 : 모델별 OOF 예측을 eeg_id 기준으로 한 표에 정렬)

- P1v2 : Nb03 Output의 `p1_oof.csv` (Input 연결)
- E1~E6 : ① Input의 `models/<EXP>/oof.csv` (권장 : nb08_pack Output = E4·E5, Nb04_train 최신 버전 = E6) → ② 업로드 `<EXP>_oof.csv` → ③ kagglehub (확인된 버전 먼저, 이후 최신부터)
- kagglehub는 Kaggle 안에서 기본 동작이 'Input 연결 후 무한 대기'라서 `DISABLE_KAGGLE_CACHE=1` + 버전당 시간 제한을 사용

In [ ]:
class _FetchTimeout(Exception):
    pass

def _on_alarm(*_):
    raise _FetchTimeout()

def internet_on(host="www.kaggle.com", port=443, timeout=5):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

_ONLINE = None
_KH_CACHE = {}                                           # 버전별 실패 사유 기록 (점검용)

def fetch_oof(exp):
    global _ONLINE
    if _ONLINE is None:
        _ONLINE = internet_on()
        if not _ONLINE:
            print("  인터넷 OFF → kagglehub 다운로드 건너뜀 (설정에서 Internet ON)")
    if not _ONLINE:
        return None
    os.environ["DISABLE_KAGGLE_CACHE"] = "1"
    try:
        import kagglehub
    except ImportError:
        print("  kagglehub 미설치 → 건너뜀")
        return None
    t0, last_err = time.time(), None
    hint = [VERSION_HINT[exp]] if exp in VERSION_HINT else []
    for v in hint + [v for v in NB04_VERSIONS if v not in hint]:      # 알려진 버전 먼저 (요청 수를 줄여 차단 방지)
        for attempt in range(3):
            signal.signal(signal.SIGALRM, _on_alarm)
            signal.alarm(FETCH_TIMEOUT)
            try:
                got = Path(kagglehub.notebook_output_download(f"{NB04_HANDLE}/versions/{v}", path=f"models/{exp}/oof.csv"))
                break
            except Exception as err:
                got, last_err = None, f"v{v} {type(err).__name__}: {str(err)[:120]}"
                if "429" in str(err) or "Too Many" in str(err):           # 요청 과다 → 잠시 쉬고 재시도
                    time.sleep(10 * (attempt + 1))
                    continue
                break
            finally:
                signal.alarm(0)
        if got is None:
            continue
        hit = got if got.is_file() else next(iter(sorted(got.glob("**/oof.csv"))), None)
        if hit is not None:
            print(f"  {exp} : kagglehub {NB04_HANDLE} v{v} | {time.time() - t0:.0f}s")
            return hit
    print(f"  {exp} : Nb04_train에서 찾지 못함 | {time.time() - t0:.0f}s | 마지막 오류 {last_err}")
    return None

def find_oof(exp):
    env = os.environ.get(f"HMS_{exp}_OOF")
    if env:
        return Path(env)
    return find_file(f"**/models/{exp}/oof.csv", f"**/{exp}_oof.csv", f"**/oof_{exp}.csv") or fetch_oof(exp)

def load_pred(path, name):
    df = pd.read_csv(path)[["eeg_id"] + PRED]
    return df.rename(columns={c: f"{name}:{c}" for c in PRED})

p1_path = Path(os.environ.get("HMS_P1_OOF") or find_file("**/profile/p1_oof.csv", "**/p1_oof.csv"))
table, models = base.copy(), []
table = table.merge(load_pred(p1_path, "P1v2"), on="eeg_id", how="left"); models.append("P1v2")
print(f"P1v2 : {p1_path}")
for exp in EXPS:
    path = find_oof(exp)
    if path is None:
        print(f"{exp} : 없음 → 건너뜀")
        continue
    table = table.merge(load_pred(path, exp), on="eeg_id", how="left")
    models.append(exp)
    print(f"{exp} : {path}")

complete = table[[f"{m}:{PRED[0]}" for m in models]].notna().all(axis=1)
table = table[complete].reset_index(drop=True)
P = {m: table[[f"{m}:{c}" for c in PRED]].to_numpy() for m in models}
assert all(np.allclose(p.sum(axis=1), 1, atol=1e-3) for p in P.values()), "예측 확률 합이 1이 아닌 모델이 있습니다"
print(f"\n정렬 완료 : 모델 {models} | 공통 {len(table):,}건 (meta {len(base):,}건 중) | 고득표 {table.high_vote.mean():.1%}")

## 2. 단독 모델 비교 (목적 : 코드북 지표 한 표 + 클래스별 고득표 KL / 판정 : H3)

- **클래스별 고득표 KL** : 고득표 구간을 전문가 최다 득표 클래스(expert_consensus)로 나눠 KL 평균 → 어떤 판정 유형에서 좋아졌는지 확인
- **H3 판정(E5가 있을 때)** : E4 대비 **LPD·LRDA 고득표 KL이 모두 낮아지고** 전체 고득표 KL이 나빠지지 않으면(+0.005 이내) 지지

In [ ]:
def row_metrics(name, pred, df=table):
    m = evaluate(df, pred)
    return {"모델": name, "KL": round(m["kl_all"], 4), "고득표 KL": round(m["kl_high"], 4),
            "편측 AUC": round(m["auc_lat"], 4), "주기성 AUC": round(m["auc_pd"], 4),
            "Seizure 민감도": round(m["sens_at_spec90"], 4)}

def kl_rows(y, p, eps=1e-15):
    y = np.clip(y, eps, 1); p = np.clip(p, eps, 1)
    return (y * np.log(y / p)).sum(axis=1)

single = pd.DataFrame([row_metrics(m, P[m]) for m in models])
display(single)

hv = table.high_vote.to_numpy().astype(bool)
Y = table[PROB_COLS].to_numpy()
cons = table.expert_consensus.to_numpy()
order = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]
per_class = pd.DataFrame({m: {c: kl_rows(Y[hv & (cons == c)], P[m][hv & (cons == c)]).mean() for c in order} for m in models}).round(4)
per_class.insert(0, "고득표 n", [int((hv & (cons == c)).sum()) for c in order])
display(per_class)

# H4 : 같은 패턴 안에서 편측↔전반을 바꿔 판정한 혼동률 (LPD↔GPD, LRDA↔GRDA / 고득표, 전문가 다수 판정 기준)
def confusion_rates(pred_prob):
    pc = np.array([c.lower() for c in ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]])[pred_prob.argmax(1)]
    t = np.array([str(c).lower() for c in cons])
    out = {}
    for a, b in [("lpd", "gpd"), ("lrda", "grda")]:
        m = hv & np.isin(t, [a, b])
        out[f"{a.upper()}↔{b.upper()}"] = float(((pc != t) & np.isin(pc, [a, b]) & m).sum() / max(m.sum(), 1))
    return out
conf = pd.DataFrame({m: confusion_rates(P[m]) for m in models if m != "P1v2"}).round(4)
print("같은 패턴 내 편측↔전반 혼동률 (고득표, 해당 두 클래스 구간 중 서로 바꿔 판정한 비율)")
display(conf)
if {"E4", "E6"} <= set(models):
    dc = conf["E6"] - conf["E4"]
    overall6 = single.set_index("모델").loc["E6", "고득표 KL"] - single.set_index("모델").loc["E4", "고득표 KL"]
    h4 = "지지" if (dc < 0).all() and overall6 <= 0.005 else ("방향 일부 일치" if (dc < 0).any() else "기각")
    print(f"H4 판정 : LPD↔GPD {dc.iloc[0]:+.4f} / LRDA↔GRDA {dc.iloc[1]:+.4f} / 전체 고득표 {overall6:+.4f} → {h4}")

if {"E4", "E5"} <= set(models):
    d = per_class["E5"] - per_class["E4"]
    overall = single.set_index("모델").loc["E5", "고득표 KL"] - single.set_index("모델").loc["E4", "고득표 KL"]
    h3 = "지지" if d["LPD"] < 0 and d["LRDA"] < 0 and overall <= 0.005 else ("방향 일부 일치" if min(d["LPD"], d["LRDA"]) < 0 else "기각")
    print(f"H3 판정 : LPD {d['LPD']:+.4f} / LRDA {d['LRDA']:+.4f} / 전체 고득표 {overall:+.4f} → {h3}")

## 3. 나중 결합 대조 (목적 : E4 개선이 '한 모델이 두 입력을 함께 본 효과'인지 확인)

- **나중 결합** : 10분 모델(E2)과 50초 모델(E3)을 따로 학습해 확률만 0.5:0.5 평균
- **초기 결합** : E4 (두 이미지를 가로로 이어 한 모델에 입력)
- **판정** : E4 고득표 KL이 나중 결합보다 낮으면 '초기 결합 효과 있음', 높으면 'E4 이득은 두 입력의 정보 합에서 나옴(앙상블로 대체 가능)'

In [ ]:
late_rows = []
if {"E2", "E3"} <= set(models):
    late = 0.5 * P["E2"] + 0.5 * P["E3"]
    P["E2+E3 평균"] = late
    late_rows = [row_metrics("0.5·E2 + 0.5·E3 (나중 결합)", late)]
    if "E4" in models:
        late_rows.append(row_metrics("E4 (초기 결합)", P["E4"]))
        k_late, k_e4 = late_rows[0]["고득표 KL"], late_rows[1]["고득표 KL"]
        print(f"초기 결합 E4 {k_e4:.4f} vs 나중 결합 {k_late:.4f} → 차이 {k_e4 - k_late:+.4f} → "
              f"{'초기 결합 효과 있음' if k_e4 < k_late else '초기 결합 효과 없음 (앙상블로 대체 가능)'}")
    display(pd.DataFrame(late_rows))
else:
    print("E2·E3 OOF가 모두 있어야 실행됩니다 → 건너뜀")

## 4. 딥러닝 + P1v2 평균 (목적 : E8 앙상블 사전 확인 / 보정 효과와 상호보완 효과 분리)

- 가중치 0.5 사전 지정 (가중치 곡선은 참고만)
- temperature scaling : fold k의 T는 나머지 fold 고득표 구간으로 맞춤 (누수 방지)
- 10/22 결론에 따라 **평균은 보정하지 않은 원본 확률로** 계산, 보정은 대조군으로만 사용

In [ ]:
def temp(p, T):
    z = np.log(np.clip(p, 1e-7, 1)) / T
    z = np.exp(z - z.max(axis=1, keepdims=True))
    return z / z.sum(axis=1, keepdims=True)

def cv_temperature(p):
    out, Ts = np.zeros_like(p), {}
    fold = table.fold.to_numpy()
    for k in np.unique(fold):
        fit = (fold != k) & hv
        Ts[int(k)] = round(float(minimize_scalar(lambda T: kl_divergence(Y[fit], temp(p[fit], T)),
                                                 bounds=(0.3, 5.0), method="bounded").x), 2)
        out[fold == k] = temp(p[fold == k], Ts[int(k)])
    return out, Ts

rows, decomp, records = [], [], []
p1 = P["P1v2"]
for exp in [m for m in models if m != "P1v2"]:
    dl = P[exp]
    dl_c, T = cv_temperature(dl)
    blend = 0.5 * dl + 0.5 * p1
    m_raw, m_T, m_bl = (evaluate(table, x) for x in (dl, dl_c, blend))
    rows += [row_metrics(f"{exp} 단독", dl), row_metrics(f"{exp} + T", dl_c), row_metrics(f"0.5·{exp} + 0.5·P1v2", blend)]
    gain_total, gain_cal = m_raw["kl_high"] - m_bl["kl_high"], m_raw["kl_high"] - m_T["kl_high"]
    gain_comp = m_T["kl_high"] - m_bl["kl_high"]
    decomp.append({"모델": exp, "fold별 T": T, "평균 이득": round(gain_total, 4), "보정 몫": round(gain_cal, 4),
                   "보정 후 남는 이득": round(gain_comp, 4),
                   "판정": "상호보완 효과 있음" if gain_comp > 0 else "효과 없음"})
    for tag, name, m, chg, verdict in [
        (f"C{exp[-1]}", f"{exp} + T", m_T, f"{exp} OOF temperature scaling (T fit on other folds, high-vote)", f"calibration gain {gain_cal:+.4f}"),
        (f"B{exp[-1]}", f"0.5*{exp} + 0.5*P1v2", m_bl, f"0.5*{exp} + 0.5*P1v2 OOF (weight pre-specified)",
         f"ensemble gain {gain_total:+.4f} (complementary {gain_comp:+.4f} after calibration)")]:
        records.append({"exp": tag, "hypothesis": "calibration control" if tag.startswith("C") else "E8 preview",
                        "change": chg, "cv_kl_all": round(m["kl_all"], 4), "cv_kl_high": round(m["kl_high"], 4),
                        "public_kl": None, "private_kl": None, "auc_lat": round(m["auc_lat"], 4), "auc_pd": round(m["auc_pd"], 4),
                        "sens_at_spec90": round(m["sens_at_spec90"], 4), "gpu_hours": 0, "kaggle_version": "nb07",
                        "verdict": verdict, "note": f"T per fold {T}"})

display(pd.DataFrame(rows))
display(pd.DataFrame(decomp))

best = max([m for m in models if m != "P1v2"], key=lambda m: -evaluate(table, P[m])["kl_high"])
curve = pd.DataFrame([{"딥러닝 비중": w, "고득표 KL": round(evaluate(table, w * P[best] + (1 - w) * p1)["kl_high"], 4)}
                      for w in np.round(np.linspace(0, 1, 11), 1)])
print(f"참고 : 가중치 곡선 ({best} + P1v2, 판정에는 사용하지 않음)")
display(curve.set_index("딥러닝 비중").T)

## 6. E8 후보 비교 (목적 : 최종 제출 조합 선택 / 모든 가중치·T는 '나머지 fold'로만 결정)

- **딥러닝 묶음 후보** : E4 단독 / 0.5·E2 + 0.5·E3 (나중 결합) / E2·E3·E4 평균 / E3·E4 평균 (E5가 있으면 E5 묶음도 추가)
- **변형 4가지** : 원본 / + T / + P1v2(가중치 중첩 선택) / + P1v2 + T
- **중첩 선택** : fold k 예측에 쓸 P1v2 가중치 w_k와 T_k는 **fold k를 뺀 나머지 fold 고득표 구간**에서만 고름 → fold k 성능은 선택에 쓰이지 않은 정직한 추정
- **최종 선택 규칙(사전 지정)** : 고득표 KL 최저 후보를 고르되, 더 단순한 후보(모델 수·P1 사용 여부)가 **0.005 이내**면 단순한 쪽 선택 (T는 비용이 없어 단순성 계산에서 제외)
- **제출 레시피** : 선택 후보의 w·T를 전체 OOF로 다시 맞춰 `e8_recipe.json`에 저장 → nb05 다중 모델 제출에 사용

In [ ]:
W_GRID = np.round(np.arange(0, 1.0001, 0.05), 2)            # 딥러닝 비중 후보 (1.0 = P1 미사용)

def kl_high_of(p, mask):
    return kl_divergence(Y[mask], p[mask])

def nested(G, use_p1, use_T):
    """fold별로 나머지 fold 고득표 구간에서 w(딥러닝 비중)와 T를 차례로 정해 해당 fold에 적용."""
    out, ws, Ts = np.zeros_like(G), {}, {}
    fold = table.fold.to_numpy()
    for k in np.unique(fold):
        tr, te = (fold != k) & hv, fold == k
        w = min(W_GRID, key=lambda w: kl_high_of(w * G + (1 - w) * p1, tr)) if use_p1 else 1.0
        mix = w * G + (1 - w) * p1
        T = float(minimize_scalar(lambda T: kl_divergence(Y[tr], temp(mix[tr], T)), bounds=(0.3, 5.0), method="bounded").x) if use_T else 1.0
        out[te] = temp(mix[te], T) if use_T else mix[te]
        ws[int(k)], Ts[int(k)] = round(float(w), 2), round(T, 2)
    return out, ws, Ts

groups = {"E4": ["E4"], "E2+E3": ["E2", "E3"], "E2+E3+E4": ["E2", "E3", "E4"], "E3+E4": ["E3", "E4"]}
if "E5" in models:
    groups.update({"E5": ["E5"], "E2+E3+E5": ["E2", "E3", "E5"], "E3+E5": ["E3", "E5"], "E4+E5": ["E4", "E5"]})
if "E6" in models:
    groups.update({"E6": ["E6"], "E4+E6": ["E4", "E6"], "E5+E6": ["E5", "E6"], "E4+E5+E6": ["E4", "E5", "E6"]})
groups = {g: ms for g, ms in groups.items() if all(m in models for m in ms)}

cand = []
for g, ms in groups.items():
    G = np.mean([P[m] for m in ms], axis=0)
    for use_p1 in (False, True):
        for use_T in (False, True):
            pred, ws, Ts = nested(G, use_p1, use_T)
            m = evaluate(table, pred)
            name = g + (" + P1v2" if use_p1 else "") + (" + T" if use_T else "")
            cand.append({"후보": name, "묶음": g, "모델 수": len(ms), "P1": use_p1, "T": use_T,
                         "단순성": len(ms) + int(use_p1), "고득표 KL": round(m["kl_high"], 4), "KL": round(m["kl_all"], 4),
                         "편측 AUC": round(m["auc_lat"], 4), "주기성 AUC": round(m["auc_pd"], 4),
                         "Seizure 민감도": round(m["sens_at_spec90"], 4),
                         "fold별 w": ws if use_p1 else "-", "fold별 T": Ts if use_T else "-"})
cand = pd.DataFrame(cand).sort_values("고득표 KL").reset_index(drop=True)
display(cand.drop(columns=["단순성"]))

best_kl = cand["고득표 KL"].min()
near = cand[cand["고득표 KL"] <= best_kl + 0.005].sort_values(["단순성", "고득표 KL"])
pick = near.iloc[0]
print(f"최저 고득표 KL : {cand.iloc[0]['후보']} {best_kl:.4f}")
print(f"선택 규칙 적용 (0.005 이내 단순 후보 우선) → E8 선택 : {pick['후보']} {pick['고득표 KL']:.4f}")

# 제출 레시피 : 선택 후보의 w·T를 전체 OOF 고득표 구간으로 다시 맞춤 (테스트에는 OOF 전체로 정한 값 사용)
ms = groups[pick["묶음"]]
G = np.mean([P[m] for m in ms], axis=0)
w_all = min(W_GRID, key=lambda w: kl_high_of(w * G + (1 - w) * p1, hv)) if pick["P1"] else 1.0
mix = w_all * G + (1 - w_all) * p1
T_all = float(minimize_scalar(lambda T: kl_divergence(Y[hv], temp(mix[hv], T)), bounds=(0.3, 5.0), method="bounded").x) if pick["T"] else 1.0
recipe = {"candidate": pick["후보"], "models": ms, "dl_weight": float(w_all), "use_p1": bool(pick["P1"]),
          "temperature": round(T_all, 4), "cv_kl_high_nested": float(pick["고득표 KL"]),
          "note": "models : 같은 비중 평균 → (P1 사용 시) w·DL + (1-w)·P1 → softmax(log p / T)"}
with open(OUT_DIR / "e8_recipe.json", "w") as f:
    json.dump(recipe, f, indent=2, ensure_ascii=False)
print("제출 레시피 :", recipe)
if recipe["use_p1"]:
    print("[주의] P1v2 포함 : 제출하려면 nb03에서 P1 fold 모델 저장 + nb05에 테스트 프로파일 지표 산출이 필요합니다.")

pk = cand.set_index("후보").loc[pick["후보"]]
records.append({"exp": "E8c", "hypothesis": "E8 candidate (nested selection)", "change": pick["후보"] + " | " + recipe["note"],
                "cv_kl_all": pk["KL"], "cv_kl_high": pk["고득표 KL"], "public_kl": None, "private_kl": None,
                "auc_lat": pk["편측 AUC"], "auc_pd": pk["주기성 AUC"], "sens_at_spec90": pk["Seizure 민감도"],
                "gpu_hours": 0, "kaggle_version": "nb07",
                "verdict": f"selected by pre-specified rule (best {cand.iloc[0]['후보']} {best_kl:.4f}, tolerance 0.005)",
                "note": f"w_all={w_all}, T_all={T_all:.3f}"})

## 7. 저장 (목적 : 비교표·결과표 행 기록)

In [ ]:
cand.to_csv(OUT_DIR / "e8_candidates.csv", index=False)
single.to_csv(OUT_DIR / "single_models.csv", index=False)
per_class.to_csv(OUT_DIR / "per_class_high_kl.csv")
pd.DataFrame(late_rows).to_csv(OUT_DIR / "late_fusion.csv", index=False)
pd.DataFrame(rows).to_csv(OUT_DIR / "blend_calibration.csv", index=False)
pd.DataFrame(decomp).to_csv(OUT_DIR / "gain_decomposition.csv", index=False)
pd.DataFrame(records).to_csv(OUT_DIR / "results_rows_nb07.csv", index=False)
print("저장 파일 :", sorted(p.name for p in OUT_DIR.iterdir()))

## 다음 단계
- 공유할 결과 : 섹션 1 정렬 줄, 섹션 2 **클래스별 고득표 KL 표**, 섹션 3 **초기·나중 결합 판정 줄**, 섹션 4 **이득 분해 표**, 섹션 6 **E8 후보 표와 선택 줄·레시피**
- E5 완료 후 다시 실행하면 H3 판정 줄이 추가로 출력됨 (지표 재계산 없이 수 분)